# Lab 05 — Choosing a transform: CWT / DWT / Hilbert

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 5 — §5.3–§5.5 (CWT/DWT, localizing a K-complex, the Hilbert envelope).

**Biomedical question.** Which representation fits THIS signal's time–frequency structure — and does each part get the *right notion of 'frequency'*?
**Task type (§1.8).** Representation (time–frequency): choose the transform to reveal the structure, not to threshold it.
**Information that must be preserved.** WHERE in time each feature occurs, and the right notion of 'frequency' for each part — a *scale* for the brief transient, a *band* for the steady rhythm, an *envelope* for the amplitude-modulated burst.
**Main assumptions.** the signal is multicomponent and non-stationary, but each part is locally narrowband enough that *some* transform localises it.
**Primary diagnostic.** put the CWT, the DWT and the Hilbert envelope side by side and read off (time, scale) of the transient, the detail level that carries it, and the burst's amplitude envelope.
**Transfer challenge.** on a real overnight EEG, *time* a sleep K-complex AND separate two nearby sleep rhythms with these same three tools.

*Self-contained: a seeded synthetic signal (fs=200 Hz, 10 s), no data files, no `bsp`; uses `pywt` + `scipy.signal`, so it runs fully offline in well under a minute. Theme (§1.8): there is **no single best transform** — CWT, DWT and Hilbert each answer a different question — **but wrong choices still exist** (a global FFT that loses WHEN; an instantaneous frequency read off a broadband mixture).*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab05_transform_choice_cwt_dwt_hilbert/lab05_transform_choice_cwt_dwt_hilbert.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab05_transform_choice_cwt_dwt_hilbert.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab05_transform_choice_cwt_dwt_hilbert.ipynb)

In [ ]:
# --- shared setup (reproducible; self-contained synthetic signal) ---
import numpy as np, matplotlib.pyplot as plt
import pywt
from scipy import signal as sig
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

FS, SECS = 200, 10.0                      # 200 Hz, 10 s

# --- the three ingredients, each a DIFFERENT kind of time-frequency structure ---
T_KC, F_KC, SIG_KC, A_KC = 2.5, 2.0, 0.25, 4.5   # (i) brief LOW-FREQ transient (K-complex-like)
F_RHY, A_RHY             = 10.0, 0.8              # (ii) steady rhythm (alpha-like), whole record
T_BURST, F_CAR           = 7.0, 40.0             # (iii) AM burst: carrier that rises and falls
SIG_B, A_BURST           = 0.6, 1.2

def synth_tf(fs=FS, secs=SECS):
    """A multicomponent, non-stationary signal with three deliberately DIFFERENT features:
      (i)  a brief LOW-FREQUENCY transient at a KNOWN instant T_KC -- a sharp, high-amplitude
           K-complex-like blip (~2 Hz, ~0.5 s): its content is a *scale/instant*, not a steady tone;
      (ii) a steady ~10 Hz RHYTHM present the whole record -- its content is a *band*;
      (iii)an amplitude-modulated BURST near T_BURST -- a 40 Hz carrier whose Gaussian envelope
           rises and falls: its content is an *envelope*, and `true_env` is the ground truth.
    No single transform reads all three well -- that is the whole point of the lab."""
    t = np.arange(int(fs*secs))/fs
    transient = A_KC*np.sin(2*np.pi*F_KC*(t - T_KC))*np.exp(-((t - T_KC)/SIG_KC)**2)
    rhythm    = A_RHY*np.sin(2*np.pi*F_RHY*t)
    true_env  = A_BURST*np.exp(-((t - T_BURST)/SIG_B)**2)     # the ground-truth modulating envelope
    burst     = true_env*np.sin(2*np.pi*F_CAR*t)
    noise     = 0.05*rng.standard_normal(t.size)
    x = transient + rhythm + burst + noise
    return t, x, true_env

t, x, true_env = synth_tf()
print(f"signal: {x.size} samples, fs={FS} Hz, {x.size/FS:.0f} s")
print(f"  (i)   transient  ~{F_KC:.0f} Hz at t={T_KC} s   (brief, low-frequency)")
print(f"  (ii)  rhythm      {F_RHY:.0f} Hz, whole record  (steady band)")
print(f"  (iii) AM burst   {F_CAR:.0f} Hz carrier near t={T_BURST} s (envelope rises & falls)")

### See it — the raw trace
Plot the whole record and zoom on the transient. Name what you can (and cannot) read off the raw trace.

In [ ]:
# --- see it: the raw trace hides most of the structure ---
fig, ax = plt.subplots(1, 2, figsize=(10, 2.8), gridspec_kw={"width_ratios": [3, 1]})
ax[0].plot(t, x, lw=0.6); ax[0].axvline(T_KC, color="r", ls="--", lw=1)
ax[0].axvspan(T_BURST-2*SIG_B, T_BURST+2*SIG_B, color="orange", alpha=0.12)
ax[0].set_xlabel("s"); ax[0].set_title("Multicomponent signal (full 10 s)")
zoom = (t > T_KC-0.7) & (t < T_KC+0.7)
ax[1].plot(t[zoom], x[zoom], lw=0.8); ax[1].axvline(T_KC, color="r", ls="--", lw=1)
ax[1].set_xlabel("s"); ax[1].set_title(f"transient @ {T_KC:.1f}s")
plt.tight_layout(); plt.show()
# Checkpoint: you can SEE a big low-frequency blip near 2.5 s and a fatter patch near 7 s -- but
# can you read the transient's frequency, WHICH band the rhythm sits in, or the burst's envelope
# shape off the raw trace? Each of the three transforms below recovers ONE of those cleanly.

## 1. CWT — localise the transient IN TIME across scales
`# TODO` The continuous wavelet transform slides a scaled wavelet over the signal, so it keeps *both* a time axis and a scale (≈frequency) axis — ideal for a one-off, low-frequency event. Compute it with `pywt.cwt` and a complex Morlet, then report *where* (time, scale, freq) the response peaks around the known transient instant.

In [ ]:
# TODO — CWT. Compute the continuous wavelet transform with pywt.cwt and a complex Morlet
#   ("cmor1.5-1.0") over LOG-spaced scales. Plot the scalogram |CWT| (time x frequency) and show
#   it localises the low-frequency transient in time. Then, in a short window around the KNOWN
#   instant T_KC, find the |CWT| maximum and report its (time, scale, freq); use pywt.cwt's
#   returned `freqs` (Hz) for the frequency axis. Set cwt_peak_time, cwt_peak_scale, cwt_peak_freq.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: the peak sits at ~2.5 s and ~2 Hz -- the CWT pins BOTH the transient's time and its
# scale. Note the response is a compact blob, not a horizontal stripe: this event is localised in
# time, unlike the steady 10 Hz rhythm which would run the full width.

## 2. DWT — which detail level carries the transient?
`# TODO` A multi-level DWT splits the signal into dyadic frequency bands (detail level *j* ≈ `[fs/2^(j+1), fs/2^j]` Hz). A brief event lands in the level whose band matches its dominant frequency / duration. Find that level by isolating each detail band and measuring its peak near the transient — then check the band actually contains ~2 Hz.

In [ ]:
# TODO — DWT. Run a multi-level DWT (pywt.wavedec, "db4", level=6). Find WHICH detail level
#   carries the transient: reconstruct each detail band on its own (zero the other coeffs +
#   waverec), take its peak |amplitude| in a short window around T_KC, and pick the LARGEST.
#   That level's band [fs/2^(j+1), fs/2^j] should contain the transient's dominant frequency
#   F_KC. Print a per-level table and set dwt_level and dwt_expected_level.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: the largest detail at the transient's instant is D6, whose band [1.56, 3.12] Hz
# brackets the ~2 Hz K-complex -- the DWT names the transient's BAND, where the CWT gave its exact
# time/scale. The 10 Hz rhythm lives two levels up (D4) and the 40 Hz carrier down at D2.

## 3. Hilbert — recover the burst's amplitude envelope
`# TODO` The AM burst's content is an *envelope*, not a tone. The analytic signal from a Hilbert transform gives `|analytic|` = the instantaneous amplitude — but only after you isolate the single component it belongs to. Band-pass around the carrier first, then take the envelope and compare it to the truth.

In [ ]:
# TODO — Hilbert. Isolate the AM burst by band-pass filtering around the carrier (a 4th-order
#   Butterworth, 30-50 Hz, zero-phase filtfilt), take the analytic signal with scipy.signal.hilbert,
#   and use |analytic| as the recovered envelope. Plot it against the TRUE modulating envelope
#   (true_env) and report their Pearson correlation. Set env_rec and corr_env.
# NOTE: an instantaneous frequency d(phase)/dt is only meaningful for a NARROWBAND, monocomponent
#   signal. Reading one off the raw broadband mixture (transient + rhythm + burst) is meaningless
#   -- that is exactly why we band-pass down to a SINGLE component before the Hilbert step.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: the recovered envelope rises and falls with the truth (corr ~0.99). The envelope is
# the RIGHT notion of 'frequency-content' for an AM burst; an instantaneous frequency here -- and
# especially off the raw multicomponent mixture -- would be a meaningless number (see the NOTE).

## 4. Live sanity check — each transform did its own job
A representation you never test is a decoration. Tie the three together with computed assertions: the DWT level must match the transient's band, the CWT max must sit at the transient's instant, and the Hilbert envelope must track the truth.

In [ ]:
# --- live sanity check: numbers computed above, not asserted on faith ---
lo, hi = FS/2**(dwt_level+1), FS/2**dwt_level
print(f"CWT : max near transient at t = {cwt_peak_time:.3f} s, freq = {cwt_peak_freq:.2f} Hz "
      f"(true t = {T_KC}, f = {F_KC:.1f})")
print(f"DWT : transient carried by D{dwt_level}, band [{lo:.2f}, {hi:.2f}] Hz (expected D{dwt_expected_level})")
print(f"Hilbert : envelope correlation = {corr_env:.4f}")

assert dwt_level == dwt_expected_level, "DWT detail level must match the transient's frequency band"
assert abs(cwt_peak_time - T_KC) < 0.15, "CWT maximum must localise the transient in time"
assert corr_env > 0.9, "recovered Hilbert envelope must track the true modulating envelope"
print("\nsanity check PASSED: transient TIMED (CWT), placed in the right BAND (DWT), "
      "envelope RECOVERED (Hilbert).")

## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the *reasoning*: matching each transform to the question it answers.

1. **Time vs band.** Which transform told you *WHEN* the transient happened, and which told you *which BAND* it lives in? Could either one, on its own, give you both the exact instant and the right frequency notion — or do they trade off?
2. **Envelope, not frequency.** For the AM burst, why is the Hilbert *envelope* meaningful while an *instantaneous frequency* is not — and what did the band-pass buy you before the Hilbert step?
3. **Evidence for a real recording.** Before trusting a K-complex *time* and a spindle *envelope* on a real overnight EEG, what would you check — sampling rate, montage/reference, artifact rejection, wavelet choice, and whether the chosen level/scale survives a *different* wavelet?

**Rule out (name the wrong transform).** Two wrong moves for THIS signal:
(a) summarising the transient with a **single global FFT** of the whole record — it confirms ~2 Hz energy exists but smears it across all 10 s, destroying *WHERE* the K-complex occurred, which breaks the **§1.8 worldview** requirement to preserve the time-location of each feature; and
(b) reading a **Hilbert instantaneous frequency off the raw broadband/multicomponent mixture** — instantaneous frequency is only defined for a narrowband, monocomponent signal, so on the mixture it yields a meaningless number, breaking the requirement to use the *right notion of frequency* for each part.
Theme (§1.8): **no single transform wins** — CWT *times* the transient, DWT *names its band*, Hilbert *tracks the burst envelope* — **but wrong choices still exist**.

> *Your answers here.*

---
*Type-2 lab for **Biomedical Signal Processing & Data Analytics**. Synthetic signal; illustrative numbers. The lesson is the method: match the transform to the question — time-location, scale/band, or envelope — not to habit.*